# Chapter 8 — Recursion and Dynamic Programming

Approaches: **bottom-up** (solve the base case, then build), **top-down** (split into subproblems), and **half-and-half** (e.g. binary search, merge sort).

DP = recursion + caching of overlapping subproblems. In Python, `functools.cache` gives top-down memoization for free. Bottom-up tables avoid deep recursion. (Python's default recursion limit is ~1000.)

In [1]:
import sys
from collections import Counter
from functools import cache

sys.setrecursionlimit(10_000)


def fib_bottom_up(n: int) -> int:
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


@cache
def fib_top_down(n: int) -> int:
    return n if n < 2 else fib_top_down(n - 1) + fib_top_down(n - 2)


assert fib_bottom_up(90) == fib_top_down(90) == 2880067194370816120
print("fib ok")

fib ok


## 8.1 Triple Step
A child climbs n stairs taking 1, 2, or 3 steps at a time. Count the distinct ways. (Grows fast; Python's big ints mean no overflow.)

In [2]:
@cache
def triple_step(n: int) -> int:
    if n < 0:
        return 0
    if n == 0:
        return 1
    return triple_step(n - 1) + triple_step(n - 2) + triple_step(n - 3)


def triple_step_iter(n: int) -> int:
    a, b, c = 0, 0, 1  # ways(n-3), ways(n-2), ways(n-1) with ways(0)=1
    for _ in range(n):
        a, b, c = b, c, a + b + c
    return c


assert [triple_step(i) for i in range(8)] == [1, 1, 2, 4, 7, 13, 24, 44]
assert all(triple_step(i) == triple_step_iter(i) for i in range(60))
print("8.1 ok")

8.1 ok


## 8.2 Robot in a Grid
A robot starts top-left in an r×c grid and may move right or down only. Some cells are off-limits. Find a path to the bottom-right. Work backward from the target and memoize failed cells → `O(rc)`.

In [3]:
def robot_path(grid: list[list[bool]]) -> list[tuple[int, int]] | None:
    """grid[r][c] is True when the cell is open."""
    if not grid or not grid[0]:
        return None
    failed: set[tuple[int, int]] = set()
    path: list[tuple[int, int]] = []

    def go(r: int, c: int) -> bool:
        if r < 0 or c < 0 or not grid[r][c] or (r, c) in failed:
            return False
        if (r, c) == (0, 0) or go(r - 1, c) or go(r, c - 1):
            path.append((r, c))
            return True
        failed.add((r, c))
        return False

    return path if go(len(grid) - 1, len(grid[0]) - 1) else None


G = [[True, True, True, False],
     [False, False, True, True],
     [True, True, False, True],
     [True, True, True, True]]
p = robot_path(G)
assert p[0] == (0, 0) and p[-1] == (3, 3)
assert all((r2 - r1, c2 - c1) in {(0, 1), (1, 0)} for (r1, c1), (r2, c2) in zip(p, p[1:]))
G[1][3] = False
assert robot_path(G) is None
print("8.2 ok")

8.2 ok


## 8.3 Magic Index
A magic index satisfies `A[i] == i`. Given a sorted array of **distinct** ints, find one with binary search. Follow-up: values not distinct. Then search both sides, but skip ranges using `min(mid-1, A[mid])` and `max(mid+1, A[mid])`.

In [4]:
def magic_index_distinct(a: list[int]) -> int | None:
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == mid:
            return mid
        if a[mid] < mid:
            lo = mid + 1
        else:
            hi = mid - 1
    return None


def magic_index_dups(a: list[int], lo: int = 0, hi: int | None = None) -> int | None:
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    if a[mid] == mid:
        return mid
    left = magic_index_dups(a, lo, min(mid - 1, a[mid]))
    if left is not None:
        return left
    return magic_index_dups(a, max(mid + 1, a[mid]), hi)


assert magic_index_distinct([-40, -20, -1, 1, 2, 3, 5, 7, 9, 12, 13]) == 7
assert magic_index_distinct([1, 2, 3]) is None
assert magic_index_dups([-10, -5, 2, 2, 2, 3, 4, 7, 9, 12, 13]) == 2
print("8.3 ok")

8.3 ok


## 8.4 Power Set
Return all subsets of a set. There are `2ⁿ` subsets, so `O(n·2ⁿ)` is optimal. Three approaches: recursion, a bitmask, and `itertools`.

In [5]:
from itertools import chain, combinations


def power_set_recursive(xs: list[int]) -> list[list[int]]:
    if not xs:
        return [[]]
    rest = power_set_recursive(xs[1:])
    return rest + [[xs[0]] + s for s in rest]


def power_set_bits(xs: list[int]) -> list[list[int]]:
    return [[x for i, x in enumerate(xs) if mask >> i & 1] for mask in range(1 << len(xs))]


def power_set_itertools(xs: list[int]) -> list[tuple[int, ...]]:
    return list(chain.from_iterable(combinations(xs, k) for k in range(len(xs) + 1)))


canon = lambda subsets: sorted(sorted(s) for s in subsets)
assert canon(power_set_recursive([1, 2, 3])) == canon(power_set_bits([1, 2, 3])) == canon(power_set_itertools([1, 2, 3]))
assert len(power_set_bits(list(range(10)))) == 1024
print("8.4 ok")

8.4 ok


## 8.5 Recursive Multiply
Multiply two positive ints without `*` or `/`, using as few operations as possible. Halve the smaller number: `a·b = 2·(a·(b//2))` (+ a if b is odd). `O(log s)`.

In [6]:
def rec_multiply(a: int, b: int) -> int:
    small, big = (a, b) if a < b else (b, a)

    def helper(s: int) -> int:
        if s == 0:
            return 0
        if s == 1:
            return big
        half = helper(s >> 1)
        return half + half + (big if s & 1 else 0)

    return helper(small)


import random
for _ in range(1000):
    x, y = random.randint(0, 10**6), random.randint(0, 10**6)
    assert rec_multiply(x, y) == x * y
print("8.5 ok")

8.5 ok


## 8.6 Towers of Hanoi
Move n disks from tower 1 to tower 3 using stacks, never placing a larger disk on a smaller one. Move n-1 disks to the buffer, move the largest, then move n-1 on top. `2ⁿ - 1` moves.

In [7]:
def hanoi(n: int) -> list[list[int]]:
    towers: list[list[int]] = [list(range(n, 0, -1)), [], []]

    def move(k: int, src: int, dst: int, buf: int) -> None:
        if k == 0:
            return
        move(k - 1, src, buf, dst)
        disk = towers[src].pop()
        assert not towers[dst] or towers[dst][-1] > disk, "illegal move"
        towers[dst].append(disk)
        move(k - 1, buf, dst, src)

    move(n, 0, 2, 1)
    return towers


assert hanoi(10) == [[], [], list(range(10, 0, -1))]
print("8.6 ok")

8.6 ok


## 8.7 Permutations without Dups
All permutations of a string of unique characters. Build by removing each char in turn and permuting the rest. `O(n² · n!)`.

In [8]:
from itertools import permutations


def perms_unique(s: str) -> list[str]:
    if len(s) <= 1:
        return [s]
    return [s[i] + p for i in range(len(s)) for p in perms_unique(s[:i] + s[i + 1:])]


assert sorted(perms_unique("abcd")) == sorted("".join(p) for p in permutations("abcd"))
print("8.7 ok")

8.7 ok


## 8.8 Permutations with Dups
Now characters may repeat; output each distinct permutation once. Recurse over a **frequency table** so duplicate branches are never generated.

In [9]:
def perms_with_dups(s: str) -> list[str]:
    counts = Counter(s)
    out: list[str] = []

    def go(prefix: list[str], remaining: int) -> None:
        if remaining == 0:
            out.append("".join(prefix))
            return
        for c in counts:
            if counts[c]:
                counts[c] -= 1
                prefix.append(c)
                go(prefix, remaining - 1)
                prefix.pop()
                counts[c] += 1

    go([], len(s))
    return out


res = perms_with_dups("aabbc")
assert len(res) == len(set(res)) == 30  # 5! / (2! 2!)
assert set(res) == {"".join(p) for p in permutations("aabbc")}
print("8.8 ok")

8.8 ok


## 8.9 Parens
Print all valid combinations of n pairs of parentheses. Add `(` while any remain, add `)` only if it wouldn't create an invalid prefix. (Count = Catalan number.)

In [10]:
def parens(n: int) -> list[str]:
    out: list[str] = []

    def go(buf: str, left: int, right: int) -> None:
        if left == right == 0:
            out.append(buf)
            return
        if left:
            go(buf + "(", left - 1, right)
        if right > left:
            go(buf + ")", left, right - 1)

    go("", n, n)
    return out


from math import comb
assert parens(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert all(len(parens(n)) == comb(2 * n, n) // (n + 1) for n in range(1, 10))
print("8.9 ok")

8.9 ok


## 8.10 Paint Fill
Given a screen (2D array of colours), a point, and a new colour, fill the surrounding same-coloured region, like a paint bucket tool. An iterative BFS avoids hitting the recursion limit on large regions.

In [11]:
from collections import deque


def paint_fill(screen: list[list[str]], r: int, c: int, new: str) -> list[list[str]]:
    old = screen[r][c]
    if old == new:
        return screen
    q = deque([(r, c)])
    screen[r][c] = new
    while q:
        cr, cc = q.popleft()
        for nr, nc in ((cr + 1, cc), (cr - 1, cc), (cr, cc + 1), (cr, cc - 1)):
            if 0 <= nr < len(screen) and 0 <= nc < len(screen[0]) and screen[nr][nc] == old:
                screen[nr][nc] = new
                q.append((nr, nc))
    return screen


img = [list(row) for row in ["RRG", "RGG", "BRR"]]
assert ["".join(r) for r in paint_fill(img, 0, 0, "Y")] == ["YYG", "YGG", "BRR"]
print("8.10 ok")

8.10 ok


## 8.11 Coins
Count the ways to make n cents from quarters, dimes, nickels, and pennies. The classic DP: iterate over coins in the outer loop so each combination is counted once. `O(n · coins)`.

In [12]:
def coin_ways(n: int, coins: tuple[int, ...] = (25, 10, 5, 1)) -> int:
    ways = [1] + [0] * n
    for coin in coins:
        for amount in range(coin, n + 1):
            ways[amount] += ways[amount - coin]
    return ways[n]


@cache
def coin_ways_recursive(n: int, index: int = 0, coins: tuple[int, ...] = (25, 10, 5, 1)) -> int:
    if index == len(coins) - 1:
        return 1  # only pennies left
    return sum(coin_ways_recursive(n - k * coins[index], index + 1, coins) for k in range(n // coins[index] + 1))


assert coin_ways(100) == coin_ways_recursive(100) == 242
print("8.11 ok")

8.11 ok


## 8.12 Eight Queens
Print every placement of 8 queens on an 8×8 board where no two share a row, column, or diagonal. Place one queen per row, and track used columns and both diagonal families in sets.

In [13]:
def n_queens(n: int = 8) -> list[list[int]]:
    """Each solution: list where solution[row] = column."""
    solutions: list[list[int]] = []
    cols: set[int] = set(); diag1: set[int] = set(); diag2: set[int] = set()
    placement: list[int] = []

    def place(row: int) -> None:
        if row == n:
            solutions.append(placement.copy())
            return
        for col in range(n):
            if col in cols or row - col in diag1 or row + col in diag2:
                continue
            cols.add(col); diag1.add(row - col); diag2.add(row + col); placement.append(col)
            place(row + 1)
            cols.remove(col); diag1.remove(row - col); diag2.remove(row + col); placement.pop()

    place(0)
    return solutions


sols = n_queens(8)
assert len(sols) == 92
print("\n".join("".join("Q" if c == col else "." for c in range(8)) for col in sols[0]))

Q.......
....Q...
.......Q
.....Q..
..Q.....
......Q.
.Q......
...Q....


## 8.13 Stack of Boxes
n boxes (w, h, d) can't be rotated. A box can go on top of another only if it is strictly smaller in all three dimensions. Find the tallest possible stack. Sort by height descending, then run a DP where `best[i]` is the tallest stack with box i at the bottom. `O(n²)`.

In [14]:
from typing import NamedTuple


class Box(NamedTuple):
    w: int
    h: int
    d: int

    def can_be_above(self, other: "Box") -> bool:
        return self.w < other.w and self.h < other.h and self.d < other.d


def tallest_stack(boxes: list[Box]) -> int:
    boxes = sorted(boxes, key=lambda b: b.h, reverse=True)
    best: list[int] = []
    for i, b in enumerate(boxes):
        # best stack with b on top = b.h + best stack of something that b can sit on
        best.append(b.h + max((best[j] for j in range(i) if b.can_be_above(boxes[j])), default=0))
    return max(best, default=0)


def tallest_brute(boxes: list[Box]) -> int:
    from itertools import permutations as perms
    best = 0
    for k in range(1, len(boxes) + 1):
        for combo in perms(boxes, k):
            if all(combo[i + 1].can_be_above(combo[i]) for i in range(k - 1)):
                best = max(best, sum(b.h for b in combo))
    return best


bxs = [Box(3, 4, 1), Box(8, 6, 2), Box(7, 8, 3), Box(2, 2, 0), Box(9, 9, 9)]
assert tallest_stack(bxs) == tallest_brute(bxs)
for _ in range(30):
    rb = [Box(*(random.randint(1, 6) for _ in range(3))) for _ in range(6)]
    assert tallest_stack(rb) == tallest_brute(rb)
print("8.13 ok, tallest =", tallest_stack(bxs))

8.13 ok, tallest = 23


## 8.14 Boolean Evaluation
Given an expression of `0`, `1`, `&`, `|`, and `^` and a desired result, count the ways to parenthesize it so it evaluates to that result. Split at each operator, count true/false ways for each side, and combine. Memoize on the substring.

In [15]:
def count_eval(expr: str, result: bool) -> int:
    @cache
    def ways(lo: int, hi: int) -> tuple[int, int]:
        """(ways_true, ways_false) for expr[lo:hi+1]."""
        if lo == hi:
            return (1, 0) if expr[lo] == "1" else (0, 1)
        t = f = 0
        for i in range(lo + 1, hi, 2):
            lt, lf = ways(lo, i - 1)
            rt, rf = ways(i + 1, hi)
            total = (lt + lf) * (rt + rf)
            match expr[i]:
                case "&": true = lt * rt
                case "|": true = lt * rt + lt * rf + lf * rt
                case "^": true = lt * rf + lf * rt
                case op: raise ValueError(op)
            t += true
            f += total - true
        return t, f

    return ways(0, len(expr) - 1)[0 if result else 1]


assert count_eval("1^0|0|1", False) == 2
assert count_eval("0&0&0&1^1|0", True) == 10
print("8.14 ok")

8.14 ok
